# Prompt Loading

The `loading.py` module loads LangChain prompt templates from configuration dictionaries or local JSON and YAML files. It supports standard prompts, few-shot prompts, and chat prompts while applying path-validation and template-format security checks.

> **Deprecated:** The prompt-loading functions in this module are deprecated since version `1.2.21` and are scheduled for removal in version `2.0.0`. Use `dumpd` or `dumps` from `langchain_core.load` for serialization and `load` or `loads` for deserialization.

## Constants

1. `URL_BASE`: Stores the base URL previously used for prompts hosted in the GitHub-based LangChain Hub.
   * **Type:**
     ```python
     str
     ```
   * **Value:**
     ```python
     "https://raw.githubusercontent.com/hwchase17/langchain-hub/master/prompts/"
     ```

2. `type_to_loader_dict`: Maps each supported prompt configuration type to its corresponding loader function.
   * **Type:**
     ```python
     dict[str, Callable[..., BasePromptTemplate[str]]]
     ```
   * **Supported keys:**
     - `"prompt"`: Loads a standard `PromptTemplate`.
     - `"few_shot"`: Loads a `FewShotPromptTemplate`.
     - `"chat"`: Loads a `ChatPromptTemplate`.

### Functions

1. `load_prompt_from_config`: Loads a prompt template from a configuration dictionary. If `_type` is absent, it defaults to `"prompt"`. File paths referenced by the configuration are validated unless dangerous paths are explicitly allowed.
   * **Deprecated:** Since version `1.2.21`; scheduled for removal in version `2.0.0`.
   * **Syntax:**
     ```python
     load_prompt_from_config(
         config: dict[str, Any], # Prompt configuration dictionary
         *,
         allow_dangerous_paths: bool = False # Whether to allow absolute paths and directory traversal
     ) -> BasePromptTemplate[str]
     ```
   * **Returns:** A prompt template created from the supplied configuration.
   * **Raises:**
     - `ValueError`: If the prompt type is unsupported.
     - `ValueError`: If a referenced path is unsafe and dangerous paths are not allowed.
     - `ValueError`: If the configuration contains an unsupported template, example, or output-parser format.

2. `load_prompt`: Loads a prompt template from a local JSON or YAML configuration file.
   * **Deprecated:** Since version `1.2.21`; scheduled for removal in version `2.0.0`.
   * **Syntax:**
     ```python
     load_prompt(
         path: str | Path, # Path to the prompt configuration file
         encoding: str | None = None, # Encoding used to read the file
         *,
         allow_dangerous_paths: bool = False # Whether to allow unsafe paths referenced by the configuration
     ) -> BasePromptTemplate[str]
     ```
   * **Supported file formats:**
     - `.json`
     - `.yaml`
     - `.yml`
   * **Returns:** A prompt template loaded from the specified file.
   * **Raises:**
     - `RuntimeError`: If the path uses the deprecated `lc://` GitHub Hub format.
     - `ValueError`: If the file type is unsupported.
     - `ValueError`: If the loaded configuration contains an unsupported prompt type or unsafe path.

## Loading Behaviour

1. **Standard prompts:** Loads the template directly or reads it from a referenced `.txt` file.
2. **Few-shot prompts:** Loads prefix and suffix templates, an example prompt, and examples stored directly or in JSON/YAML files.
3. **Chat prompts:** Builds a `ChatPromptTemplate` from the first message template in the configuration.
4. **Output parsers:** Supports the default string output parser through `StrOutputParser`.
5. **Template formats:** Rejects loading `jinja2` templates because they can permit arbitrary code execution.
6. **Path validation:** Rejects absolute paths and paths containing `..` unless `allow_dangerous_paths=True` is supplied.
7. **Template files:** Only `.txt` files are supported when a template is loaded through a referenced path.